In [2]:
import os
import json

In [5]:
base_dir = os.path.dirname((os.getcwd()))
base_dir

'c:\\Users\\HOANG QUOC CUONG\\OneDrive\\Máy tính\\important project\\Workflow-Aware-Agent-Distillation'

In [7]:
bugfix_dir = os.path.join(base_dir, 'teacher_dataset', 'by_workflow', 'bugfixing')
review_response_dir = os.path.join(base_dir, 'teacher_dataset', 'by_workflow', 'review_response')
style_review_dir = os.path.join(base_dir, 'teacher_dataset', 'by_workflow', 'style_review')
test_gen_dir = os.path.join(base_dir, 'teacher_dataset', 'by_workflow', 'test_generation')

In [27]:
cpp_bugfix_dir = os.path.join(bugfix_dir, 'cpp', 'records.jsonl')
python_bugfix_dir = os.path.join(bugfix_dir, 'python', 'records.jsonl')
java_bugfix_dir = os.path.join(bugfix_dir, 'java', 'records.jsonl')

cpp_review_response_dir = os.path.join(review_response_dir, 'cpp', 'records.jsonl')
python_review_response_dir = os.path.join(review_response_dir, 'python', 'records.jsonl')
java_review_response_dir = os.path.join(review_response_dir, 'java', 'records.jsonl')

cpp_style_review_dir = os.path.join(style_review_dir, 'cpp', 'records.jsonl')
python_style_review_dir = os.path.join(style_review_dir, 'python', 'records.jsonl')
java_style_review_dir = os.path.join(style_review_dir, 'java', 'records.jsonl')

cpp_test_gen_dir = os.path.join(test_gen_dir, 'cpp', 'records.jsonl')
python_test_gen_dir = os.path.join(test_gen_dir, 'python', 'records.jsonl')
java_test_gen_dir = os.path.join(test_gen_dir, 'java', 'records.jsonl')

In [22]:
with open (python_bugfix_dir, 'r', encoding = 'utf-8') as f:
    python_bugfix_trajectories = [json.loads(item) for item in f]

with open (cpp_bugfix_dir, 'r', encoding = 'utf-8') as f:
    cpp_bugfix_trajectories = [json.loads(item) for item in f]

with open (java_bugfix_dir, 'r', encoding = 'utf-8') as f:
    java_bugfix_trajectories = [json.loads(item) for item in f]

In [30]:
with open(python_test_gen_dir, "r", encoding="utf-8") as f:
    python_test_gen_trajectories = [
        json.loads(item) for item in f if item.strip()
    ]

with open(cpp_test_gen_dir, "r", encoding="utf-8") as f:
    cpp_test_gen_trajectories = [json.loads(item) for item in f if item.strip()]

with open(java_test_gen_dir, "r", encoding="utf-8") as f:
    java_test_gen_trajectories = [
        json.loads(item) for item in f if item.strip()
    ]

In [28]:
with open(python_review_response_dir, "r", encoding="utf-8") as f:
    python_review_response_trajectories = [
        json.loads(item) for item in f if item.strip()
    ]

with open(cpp_review_response_dir, "r", encoding="utf-8") as f:
    cpp_review_response_trajectories = [
        json.loads(item) for item in f if item.strip()
    ]

with open(java_review_response_dir, "r", encoding="utf-8") as f:
    java_review_response_trajectories = [
        json.loads(item) for item in f if item.strip()
    ]

In [29]:
with open(python_style_review_dir, "r", encoding="utf-8") as f:
    python_style_review_trajectories = [
        json.loads(item) for item in f if item.strip()
    ]

with open(cpp_style_review_dir, "r", encoding="utf-8") as f:
    cpp_style_review_trajectories = [
        json.loads(item) for item in f if item.strip()
    ]

with open(java_style_review_dir, "r", encoding="utf-8") as f:
    java_style_review_trajectories = [
        json.loads(item) for item in f if item.strip()
    ]

In [20]:
python_bugfix_trajectories[0].keys()

dict_keys(['api_calls', 'cost', 'final_patch', 'language', 'runtime', 'status', 'success', 'task_id', 'tokens_input', 'tokens_output', 'trajectory', 'verifier_score', 'workflow'])

1 task instance (cùng repo cùng state cụ thể, cùng task cụ thể) được xác định bởi (task_id, workflow)

final_patch: agent có để lại bản diff code mà collector đọc được không?


status: lần chạy kết thúc như thế nào? Ví dụ completed_success nghĩa là agent đã gọi submit và có patch không rỗng; timeout nghĩa là hết thời gian.


success: kết quả đánh giá. Nếu đã có verifier, trường này lấy từ verifier. Nếu chưa có, collector tạm suy ra từ status.

In [25]:
len(python_bugfix_trajectories), len(cpp_bugfix_trajectories), len(java_bugfix_trajectories)

(275, 138, 109)

In [24]:
275 + 138 + 109

522

## Tổng hợp theo workflow và ngôn ngữ

`records` đếm mọi dòng JSONL; `trajectories` chỉ đếm record có `trajectory.history` không rỗng. `submitted_patch_runs` đếm các lần chạy có `status == 'completed_success'` và `final_patch` không rỗng. `submitted_patch_tasks` đếm `task_id` duy nhất trong từng nhóm workflow/ngôn ngữ. Dòng TOTAL cộng các nhóm, nên cùng `task_id` ở hai workflow được tính là hai task theo workflow. Đây là số task đã submit patch, chưa phải số task pass verifier. Dữ liệu trong `by_workflow` là bản tách tại thời điểm chạy script tách records; hãy tách lại nếu `records.jsonl` gốc đã có thêm dòng.

In [ ]:
from pathlib import Path
import pandas as pd

records_root = Path(base_dir) / 'teacher_dataset' / 'by_workflow'
summary_rows = []

for records_path in sorted(records_root.glob('*/*/records.jsonl')):
    workflow, language = records_path.parent.parent.name, records_path.parent.name
    records = trajectories = submitted_patch_runs = 0
    task_ids = set()
    submitted_task_ids = set()

    with records_path.open('r', encoding='utf-8') as handle:
        for line in handle:
            if not line.strip():
                continue
            record = json.loads(line)
            records += 1
            task_id = record.get('task_id')
            if task_id is not None:
                task_ids.add(task_id)
            trajectory = record.get('trajectory')
            trajectories += isinstance(trajectory, dict) and bool(trajectory.get('history'))
            patch = record.get('final_patch')
            if record.get('status') == 'completed_success' and isinstance(patch, str) and patch.strip():
                submitted_patch_runs += 1
                if task_id is not None:
                    submitted_task_ids.add(task_id)

    summary_rows.append({
        'workflow': workflow,
        'language': language,
        'records': records,
        'trajectories': trajectories,
        'unique_tasks': len(task_ids),
        'submitted_patch_runs': submitted_patch_runs,
        'submitted_patch_tasks': len(submitted_task_ids),
    })

summary = pd.DataFrame(summary_rows)
if summary.empty:
    raise FileNotFoundError(f'Không tìm thấy records.jsonl trong {records_root}')
display(summary)

In [ ]:
metric_columns = [
    'records', 'trajectories', 'unique_tasks',
    'submitted_patch_runs', 'submitted_patch_tasks',
]
by_workflow = summary.groupby('workflow', as_index=False)[metric_columns].sum()
overall = summary[metric_columns].sum().to_frame().T
overall.insert(0, 'workflow', 'TOTAL')
display(pd.concat([by_workflow, overall], ignore_index=True))